In [1]:
#Import packages:

from pathlib import Path
import pandas as pd
import numpy as np
import os
import glob



To get and copy the path of a folder:

Method 1: Drag and Drop Open Finder. Find the folder you want. Open Terminal. Drag the folder from Finder into the Terminal window.

Method 2: Copy Pathname In Finder, right-click the folder. Hold the Option (⌥) key. You'll see "Copy 'FolderName' as Pathname"

In [20]:
#Read files:

GBM_PATH =  "/Users/olahassan/Desktop/Python/TCGA_GBM"

TUMOR_PATH = os.path.join(GBM_PATH, "Tumor")

NORMAL_PATH = os.path.join(GBM_PATH, "Normal")

OUTPUT_PATH = "/Users/olahassan/Desktop/Python/TCGA_GBM/results"

#add a apath to the project  

GBM_PROJECT = Path("/Users/olahassan/Desktop/Python/TCGA_GBM")



In [22]:
#validate files exist

print(os.listdir(GBM_PATH))

['.DS_Store', 'data_processed', 'Tumor', 'results', 'Normal', 'notebooks']


In [24]:
#Convert tumor files into a vector
tumor_files = glob.glob(
    os.path.join(TUMOR_PATH, "*", "*.tsv"),
recursive=True
)
print(len(tumor_files))




5


In [26]:
#Convert normal files into a vector

normal_files = glob.glob(
    os.path.join(NORMAL_PATH, "**", "*.tsv"),
    recursive=True
)
print(len(normal_files))

5


In [28]:
all_files = tumor_files + normal_files

len(all_files)

10

In [30]:
from pathlib import Path

all_files = [
    Path(f)
    for f in all_files
]

In [32]:
type(all_files[0])

pathlib.PosixPath

In [34]:
count_matrix = None
sample_info = []

In [36]:
#Create count matrix, 
#Use gene_id
#Remove Ensembl version
#Collapse duplicates BEFORE merging
#Keep only raw counts



for file in all_files:

    sample_id = file.parent.name

    condition = file.parent.parent.name

    print(condition, sample_id)

    df = pd.read_csv(
        file,
        sep="\t",
        comment="#"
    )

    df["gene_id"] = (
        df["gene_id"]
        .str.split(".")
        .str[0]
    )

    df = df[
        [
            "gene_id",
            "unstranded"
        ]
    ]

    # collapse duplicate gene IDs
    df = (
        df
        .groupby("gene_id")["unstranded"]
        .sum()
        .to_frame()
    )

    df.columns = [sample_id]


    if count_matrix is None:
        count_matrix = df
    else:
        count_matrix = count_matrix.join(
            df,
            how="inner"
        )


    sample_info.append(
        {
            "sample": sample_id,
            "condition": condition
        }
    )

Tumor 77c261f0-384c-4906-a57e-7c3b35c17041
Tumor 99ff2016-a97a-4142-b70a-1bb59d6264e9
Tumor d4649bc7-09d2-4d90-bafd-0bec72429b00
Tumor afbd1975-263b-404b-a738-886bf25c0223
Tumor 444a2332-9e52-4cf8-94d9-07a8b829633e
Normal 64a2ddbe-3614-409a-af16-170a1471db2a
Normal 4bed9101-07f4-4d76-b79f-6eb8de04bd19
Normal ddaed459-ab19-4f75-a1a9-72761287c6f5
Normal 477a4ae1-84c0-49f2-b3b3-4015b8c26f18
Normal c410d37e-e445-4f5f-8937-e6796dd52364


In [38]:
count_matrix.shape

(60620, 10)

In [40]:
count_matrix.head()

,77c261f0-384c-4906-a57e-7c3b35c17041,99ff2016-a97a-4142-b70a-1bb59d6264e9,d4649bc7-09d2-4d90-bafd-0bec72429b00,afbd1975-263b-404b-a738-886bf25c0223,444a2332-9e52-4cf8-94d9-07a8b829633e,64a2ddbe-3614-409a-af16-170a1471db2a,4bed9101-07f4-4d76-b79f-6eb8de04bd19,ddaed459-ab19-4f75-a1a9-72761287c6f5,477a4ae1-84c0-49f2-b3b3-4015b8c26f18,c410d37e-e445-4f5f-8937-e6796dd52364
gene_id,,,,,,,,,,
ENSG00000000003,2345,1550,2934,2420,2316,699,608,452,413,546
ENSG00000000005,9,3,10,2,7,5,7,5,0,9
ENSG00000000419,1843,1180,762,676,1059,971,986,928,795,1074
ENSG00000000457,606,748,950,697,1065,606,492,402,532,478
ENSG00000000460,340,434,442,364,783,81,81,61,146,101


In [42]:
#Check for duplicates

count_matrix.index.duplicated().sum()

np.int64(0)

In [44]:
#Create folder:
(GBM_PROJECT / "data_processed").mkdir( exist_ok=True)


In [93]:
#Save counts

count_matrix.to_csv(
    GBM_PROJECT / "data_processed" / "count_matrix_ensembl.csv"
)

In [46]:
#save metadata

metadata = pd.DataFrame(sample_info)

metadata = metadata.set_index("sample")

In [48]:
#check 
metadata.shape

(10, 1)

In [50]:
#save
metadata.to_csv(
    GBM_PROJECT / "data_processed" / "metadata.csv"
)

In [52]:
# Create another count matrix file with gene symbols istad of ensemble IDs, to use it for DESEq later

Tumor_file = glob.glob(
    os.path.join(TUMOR_PATH, "*", "*.tsv")
)[0]


Tumor_gene_map = pd.read_csv(
    Tumor_file,
    sep="\t",
    comment="#"
)[["gene_id","gene_name"]]

#Remove un needed columns 
Tumor_gene_map = Tumor_gene_map[
    Tumor_gene_map["gene_id"].str.startswith("ENSG")
]


#Remove Ensemble version numbers
Tumor_gene_map["ensembl_id"] = (
    Tumor_gene_map["gene_id"]
    .str.split(".")
    .str[0]
)

Tumor_gene_map.head()

,gene_id,gene_name,ensembl_id
4,ENSG00000000003.15,TSPAN6,ENSG00000000003
5,ENSG00000000005.6,TNMD,ENSG00000000005
6,ENSG00000000419.13,DPM1,ENSG00000000419
7,ENSG00000000457.14,SCYL3,ENSG00000000457
8,ENSG00000000460.17,C1orf112,ENSG00000000460


In [54]:
#Repeat for Normal

Normal_file = glob.glob(
    os.path.join(NORMAL_PATH, "*", "*.tsv")
)[0]


Normal_gene_map = pd.read_csv(
    Normal_file,
    sep="\t",
    comment="#"
)[["gene_id","gene_name"]]


Normal_gene_map = Normal_gene_map[
    Normal_gene_map["gene_id"].str.startswith("ENSG")
]


Normal_gene_map["ensembl_id"] = (
    Normal_gene_map["gene_id"]
    .str.split(".")
    .str[0]
)

Normal_gene_map.head()

,gene_id,gene_name,ensembl_id
4,ENSG00000000003.15,TSPAN6,ENSG00000000003
5,ENSG00000000005.6,TNMD,ENSG00000000005
6,ENSG00000000419.13,DPM1,ENSG00000000419
7,ENSG00000000457.14,SCYL3,ENSG00000000457
8,ENSG00000000460.17,C1orf112,ENSG00000000460


In [56]:
#Check Tumor and Normal mappings are identical (Very important)

Tumor_gene_map.shape, Normal_gene_map.shape

((60660, 3), (60660, 3))

In [58]:
(
Tumor_gene_map["ensembl_id"]
==
Normal_gene_map["ensembl_id"]
).all()

np.True_

In [70]:
#Create final annotation table

#using only one of the  files Tumor or Normal to create the final annotation table is correct,
#provided that Tumor and Normal files were generated from the same GDC workflow (STAR - Counts) and the same GENCODE reference version.
#since both are TCGA GDC files, they should both use the same annotation:

gene_annotation = Tumor_gene_map[
    ["ensembl_id","gene_name"]
].copy()


In [72]:
#rename

gene_annotation.columns = [
    "ensembl_id",
    "gene_symbol"
]

In [64]:
gene_annotation.to_csv(GBM_PROJECT / "data_processed" / 
    "gene_annotation_GENCODE.csv",
    index=False
)

In [74]:
# Apply this to the count matrix 

#Remove older version of the count matrix
count_matrix.index = (
    count_matrix.index
    .str.split(".")
    .str[0]
)

#Then merge annotation and Set gene symbols as index

count_matrix_symbol = (
    count_matrix
    .reset_index()
    .merge(
        gene_annotation,
        left_on="gene_id",
        right_on="ensembl_id",
        how="left"
    )
)


In [76]:
count_matrix_symbol.head()

,gene_id,77c261f0-384c-4906-a57e-7c3b35c17041,99ff2016-a97a-4142-b70a-1bb59d6264e9,d4649bc7-09d2-4d90-bafd-0bec72429b00,afbd1975-263b-404b-a738-886bf25c0223,444a2332-9e52-4cf8-94d9-07a8b829633e,64a2ddbe-3614-409a-af16-170a1471db2a,4bed9101-07f4-4d76-b79f-6eb8de04bd19,ddaed459-ab19-4f75-a1a9-72761287c6f5,477a4ae1-84c0-49f2-b3b3-4015b8c26f18,c410d37e-e445-4f5f-8937-e6796dd52364,ensembl_id,gene_symbol
0,ENSG00000000003,2345,1550,2934,2420,2316,699,608,452,413,546,ENSG00000000003,TSPAN6
1,ENSG00000000005,9,3,10,2,7,5,7,5,0,9,ENSG00000000005,TNMD
2,ENSG00000000419,1843,1180,762,676,1059,971,986,928,795,1074,ENSG00000000419,DPM1
3,ENSG00000000457,606,748,950,697,1065,606,492,402,532,478,ENSG00000000457,SCYL3
4,ENSG00000000460,340,434,442,364,783,81,81,61,146,101,ENSG00000000460,C1orf112


In [78]:
#Handle duplicate gene symbols (This is important., DESeq2 needs unique rows.)


count_matrix_symbol = (
    count_matrix_symbol
    .groupby(count_matrix_symbol.index)
    .sum()
)

In [80]:
count_matrix_symbol.shape

(60664, 13)

In [82]:
count_matrix_symbol.head()

,gene_id,77c261f0-384c-4906-a57e-7c3b35c17041,99ff2016-a97a-4142-b70a-1bb59d6264e9,d4649bc7-09d2-4d90-bafd-0bec72429b00,afbd1975-263b-404b-a738-886bf25c0223,444a2332-9e52-4cf8-94d9-07a8b829633e,64a2ddbe-3614-409a-af16-170a1471db2a,4bed9101-07f4-4d76-b79f-6eb8de04bd19,ddaed459-ab19-4f75-a1a9-72761287c6f5,477a4ae1-84c0-49f2-b3b3-4015b8c26f18,c410d37e-e445-4f5f-8937-e6796dd52364,ensembl_id,gene_symbol
0,ENSG00000000003,2345,1550,2934,2420,2316,699,608,452,413,546,ENSG00000000003,TSPAN6
1,ENSG00000000005,9,3,10,2,7,5,7,5,0,9,ENSG00000000005,TNMD
2,ENSG00000000419,1843,1180,762,676,1059,971,986,928,795,1074,ENSG00000000419,DPM1
3,ENSG00000000457,606,748,950,697,1065,606,492,402,532,478,ENSG00000000457,SCYL3
4,ENSG00000000460,340,434,442,364,783,81,81,61,146,101,ENSG00000000460,C1orf112


In [152]:
#Save counts having the gene symbols and the ensembl Ids together

count_matrix_symbol.to_csv(
    GBM_PROJECT / "data_processed" / "count_matrix_ensembe_symbol.csv"
)

Now the work flow should be as follows 


count_matrix_ensembl.csv
        |
        ↓
Notebook 2 QC
        |
        ↓
Notebook 3 PyDESeq2
        |
        ↓
Merge DESeq2 results with gene_annotation
        |
        ↓
Add gene symbols for plots